In [1]:
from pyMoove import io, plotting, calculations, cleaning, statistics, displacement

from scripts.zone_info import get_zone_info
from pathlib import Path

# Resolve data directory (works from project root or scripts directory)
DATA_DIR = Path("data")


In [2]:

data = io.load_processed_dataset(DATA_DIR)
zones_path = DATA_DIR / "zone_info.csv"
zones = get_zone_info(zones_path)


Loaded dataset in folder: data
  positions Memory: 571.74 MiB
  feeding Memory: 0.28 MiB
  replacements Memory: 0.02 MiB
Total memory: 572.05 MiB



In [3]:

# Calculate statistics for all feeding bins
#bins = sorted(data["feeding"]["bin"].dropna().unique())
#bin_statistics = calculations.calculate_bin_position_statistics(data, bins, outlier_confidence=0.9999)

# Save results to Parquet
bin_statistics_path = DATA_DIR / "bin_position_stats.parquet"
#bin_statistics.to_parquet(bin_statistics_path, index=False)

bin_statistics = io.load_bin_stats_parquet(bin_statistics_path)
#bin_statistics

In [4]:
stats = statistics.inspect_replacement_event_statistics(data['positions'], data['replacements'], bin_statistics)
data['replacements'] = cleaning.filter_replacement_events(data['replacements'], stats)

Total event before cleaning: 550
Events retained: 466
Expected removal: 84
Events removed: 84


In [ ]:
replacements = data['replacements']
for bin_id in range(11, 21):
    bin_replacements = replacements.loc[replacements['bin']==bin_id]
    bin_replacements.sort_values('time').reset_index(drop=True)
    #detected_replacements = displacement.detect_replacements_by_bin_replacement(data['positions'], bin_statistics, bin_id, max_bin_distance=1.5)
    detected_replacements = displacement.detect_replacements_by_bin_hug(data['positions'], bin_statistics, bin_id, max_bin_distance=2.0)
    recall = displacement.check_replacement_recall(bin_replacements, detected_replacements, acceptance_interval=60, cow_role_matters=False, show_detections=True)

Actual replacements: 9
Detected replacements: 4645
8 / 9 replacement events were detected
Actual replacements: 4
Detected replacements: 4204
3 / 4 replacement events were detected
Actual replacements: 2
Detected replacements: 3409
1 / 2 replacement events were detected
Actual replacements: 3
Detected replacements: 2820
1 / 3 replacement events were detected
Actual replacements: 4
Detected replacements: 2719
2 / 4 replacement events were detected
Actual replacements: 10
Detected replacements: 2678
6 / 10 replacement events were detected
Actual replacements: 6
Detected replacements: 2617
6 / 6 replacement events were detected
Actual replacements: 4
Detected replacements: 2189
3 / 4 replacement events were detected
Actual replacements: 4
Detected replacements: 1709
2 / 4 replacement events were detected
Actual replacements: 4
Detected replacements: 727
2 / 4 replacement events were detected


In [ ]:
detected_replacements.loc[105:115]

In [ ]:
animations = []
for i in range(0, 4):
    fig, ax, animation = plotting.plot_replacement_example(data['positions'], bin_replacements, zones=zones, bin_statistics=bin_statistics, event_index=i, animate=True, relative_start_time=30, relative_end_time=30)
    animations.append(animation)

In [ ]:

animations = []
for i in range(110, 112):
    fig, ax, animation = plotting.plot_replacement_example(data['positions'], detected_replacements, zones=zones, bin_statistics=bin_statistics, event_index=i, animate=True, relative_start_time=30, relative_end_time=30)
    animations.append(animation)

In [ ]:
bins = range(1,59)
plotting.plot_bins(zones, bin_statistics, bins)

In [ ]:
fig, ax = plotting.plot_feeding_event(data, zones, event_index=0, bin_statistics=bin_statistics)

In [ ]:

# Option A: Plot all feeding events for a specific bin (e.g. Bin 21, 24, etc.)
bins = range(58, 59)
for i in bins:
    target_bin = i
    fig, ax = plotting.plot_bin_feeding_positions(data, zones, bin_id=target_bin, color_by_cow=True, outlier_confidence=0.9999)

# Option B: Plot a single feeding event by index (uncomment to use):
# fig, ax = plot_feeding_event(data, zones, event_index=0)